# 📖 Notebook 2: News Feed Generation

When you open Instagram, your feed loads in under 500ms — even though you might follow 1,000 people  
who collectively posted hundreds of times today. How?

The answer is **pre-computed feeds**. Instead of assembling your feed when you open the app,  
Instagram builds it in advance every time someone you follow posts.

## Learning Objectives

By the end of this notebook, you'll understand:
- **Fan-out on Read** — assembling the feed at read time (simple but slow)
- **Fan-out on Write** — pre-computing feeds when posts are created (fast reads)
- **The Celebrity Problem** — why fan-out on write breaks for users with millions of followers
- **Hybrid Approach** — how Instagram combines both strategies in production
- How **Redis sorted sets** store precomputed feeds

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/instagram
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `instagram_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import time
import json

# ── Connections ───────────────────────────────────────────
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "instagram_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ PostgreSQL — {cur.fetchone()[0]} users")
    cur.execute("SELECT COUNT(*) FROM posts")
    print(f"   {cur.fetchone()[0]} posts")
    cur.execute("SELECT COUNT(*) FROM follows")
    print(f"   {cur.fetchone()[0]} follow relationships")
    cur.execute("SELECT COUNT(*) FROM precomputed_feed")
    print(f"   {cur.fetchone()[0]} precomputed feed entries")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print(f"✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Core Problem

When User 1 opens their feed, we need to show them recent posts from everyone they follow.

```
User 1 follows: user2, user7, user15, celeb_alice, celeb_bob

We need to:
  1. Find all people User 1 follows
  2. Get recent posts from each of them
  3. Merge and sort by time
  4. Return the top 20

Now imagine 500 MILLION users doing this simultaneously.
```

There are two fundamentally different approaches.

## Strategy 1: Fan-Out on Read ("Pull Model")

**When the user opens their feed**, we:
1. Look up everyone they follow
2. Fetch recent posts from each followed user
3. Merge and sort by timestamp
4. Return the top N

```
User opens feed
      │
      ▼
┌─────────────┐     ┌──────────────┐
│ Get followed │────►│ For EACH     │──► Merge + Sort ──► Return top 20
│ user IDs     │     │ followed user│
│ (1 query)    │     │ get posts    │
│              │     │ (N queries)  │
└─────────────┘     └──────────────┘
```

Let's implement this and see how it performs.

In [ ]:
def get_feed_fan_out_on_read(user_id: int, limit: int = 20) -> list:
    """
    Fan-Out on Read: Build the feed at read time.
    
    For each user we follow, query their recent posts,
    then merge everything together sorted by time.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()

    # Step 1: Get all users this person follows
    cur.execute(
        "SELECT followee_id FROM follows WHERE follower_id = %s",
        (user_id,)
    )
    followees = [row["followee_id"] for row in cur.fetchall()]
    step1_time = time.time() - start

    if not followees:
        conn.close()
        return []

    # Step 2: Get recent posts from EACH followed user (one query per user)
    all_posts = []
    for followee_id in followees:
        cur.execute(
            """SELECT id, author_id, caption, like_count, created_at
               FROM posts
               WHERE author_id = %s
               ORDER BY created_at DESC
               LIMIT 10""",
            (followee_id,)
        )
        all_posts.extend(cur.fetchall())
    step2_time = time.time() - start - step1_time

    # Step 3: Sort by time (newest first) and take top N
    all_posts.sort(key=lambda p: p["created_at"], reverse=True)
    feed = all_posts[:limit]
    total_time = time.time() - start

    conn.close()

    print(f"📊 Fan-out on READ stats:")
    print(f"   Following: {len(followees)} users")
    print(f"   Queries: 1 (follows) + {len(followees)} (posts) = {1 + len(followees)} total")
    print(f"   Posts fetched: {len(all_posts)}")
    print(f"   Time: {total_time*1000:.1f}ms (follows: {step1_time*1000:.1f}ms, posts: {step2_time*1000:.1f}ms)")

    return feed

# Let's get User 1's feed using fan-out on read
print("Loading User 1's feed (fan-out on read)...\n")
feed = get_feed_fan_out_on_read(user_id=1, limit=10)
print(f"\nTop 10 posts in feed:")
for i, post in enumerate(feed, 1):
    print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}...")

### Why Fan-Out on Read Doesn't Scale

With our 53 users, it's fast. But imagine Instagram's real scale:

| Metric | Our Demo | Real Instagram |
|--------|----------|---------------|
| Users | 53 | 500,000,000 |
| Avg follows | ~10 | ~500 |
| Feed requests/sec | 1 | 150,000+ |
| DB queries per feed | ~11 | ~501 |

At 150,000 feed requests/second × 501 queries each = **75 million queries/second**.  
That's absurd. No database can handle that.

**The core issue**: we're doing all the expensive work at read time — exactly when users expect instant results.

## Strategy 2: Fan-Out on Write ("Push Model")

Instead of building the feed when the user reads it, we build it when someone **posts**.

When User 7 creates a new post:
1. Save the post to the database (as before)
2. Look up everyone who follows User 7
3. Push this post into each follower's precomputed feed

```
User 7 posts
      │
      ▼
┌─────────────┐     ┌──────────────────┐
│ Save post   │────►│ Get followers    │
│ to database │     │ of User 7        │
└─────────────┘     └────────┬─────────┘
                             │
                    ┌────────▼─────────┐
                    │ Push post into   │
                    │ EACH follower's  │
                    │ precomputed feed │
                    └──────────────────┘
                        │    │    │
                        ▼    ▼    ▼
                    feed:1 feed:3 feed:15  (Redis sorted sets)
```

Now when the user opens their feed, we just read their precomputed feed — **one operation**.

In [ ]:
def fan_out_on_write(author_id: int, post_id: int, created_at_ts: float):
    """
    Fan-Out on Write: When a post is created, push it to all followers' feeds.

    This runs ASYNCHRONOUSLY after the post is saved — the user doesn't wait.
    We use Redis sorted sets (ZSET) where:
    - Key = feed:{user_id}
    - Member = post_id
    - Score = timestamp (for chronological ordering)
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()
    start = time.time()

    # Get all followers of this author
    cur.execute(
        "SELECT follower_id FROM follows WHERE followee_id = %s",
        (author_id,)
    )
    followers = [row[0] for row in cur.fetchall()]

    # Push post_id into each follower's Redis sorted set
    pipeline = r.pipeline()  # batch Redis commands for efficiency
    for follower_id in followers:
        feed_key = f"feed:{follower_id}"
        pipeline.zadd(feed_key, {str(post_id): created_at_ts})
        # Keep only the most recent 500 posts per feed
        pipeline.zremrangebyrank(feed_key, 0, -501)
    pipeline.execute()

    elapsed = (time.time() - start) * 1000
    conn.close()

    print(f"📤 Fan-out on WRITE:")
    print(f"   Post #{post_id} by user {author_id}")
    print(f"   Pushed to {len(followers)} followers' feeds")
    print(f"   Time: {elapsed:.1f}ms")
    return followers

# Simulate User 5 creating a new post
conn = get_db()
cur = conn.cursor()
cur.execute(
    """INSERT INTO posts (author_id, caption, media_type, media_key, created_at)
       VALUES (5, 'Fan-out demo post! 🚀', 'photo', 'photos/user_5/fanout_demo.jpg', NOW())
       RETURNING id, extract(epoch from created_at)"""
)
new_post_id, new_post_ts = cur.fetchone()
new_post_ts = float(new_post_ts)  # psycopg2 returns Decimal; Redis needs float
conn.commit()
conn.close()

print(f"Created post #{new_post_id}\n")

# Now fan it out!
followers = fan_out_on_write(author_id=5, post_id=new_post_id, created_at_ts=new_post_ts)

### Reading the Precomputed Feed

Now reading the feed is **dead simple** — just read from Redis.
One operation, sub-millisecond.

In [ ]:
def get_feed_fan_out_on_write(user_id: int, limit: int = 20) -> list:
    """
    Read the precomputed feed from Redis.
    Then "hydrate" each post_id with metadata from PostgreSQL.
    """
    r = get_redis()
    start = time.time()

    # Get top N post IDs from Redis (sorted by score = timestamp, descending)
    feed_key = f"feed:{user_id}"
    post_ids = r.zrevrange(feed_key, 0, limit - 1)
    redis_time = (time.time() - start) * 1000

    if not post_ids:
        print(f"Feed for user {user_id} is empty in Redis (not yet populated)")
        return []

    # Hydrate: fetch full post data from PostgreSQL
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cur.execute(
        """SELECT id, author_id, caption, like_count, created_at
           FROM posts
           WHERE id = ANY(%s)
           ORDER BY created_at DESC""",
        ([int(pid) for pid in post_ids],)
    )
    posts = cur.fetchall()
    total_time = (time.time() - start) * 1000
    conn.close()

    print(f"📊 Fan-out on WRITE read stats:")
    print(f"   Redis lookup: {redis_time:.1f}ms ({len(post_ids)} post IDs)")
    print(f"   Hydration query: 1 (batch fetch by IDs)")
    print(f"   Total: {total_time:.1f}ms")

    return posts

# Read the feed for a follower of User 5
# First, find someone who follows User 5
if followers:
    test_user = followers[0]
    print(f"Reading feed for user {test_user} (follows user 5)...\n")
    feed = get_feed_fan_out_on_write(user_id=test_user, limit=10)
    if feed:
        print(f"\nTop posts in feed:")
        for i, post in enumerate(feed, 1):
            print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}")

### Comparing the Two Approaches

| Metric | Fan-out on Read | Fan-out on Write |
|--------|----------------|------------------|
| **Read speed** | Slow (N+1 queries) | Fast (1 Redis read + 1 batch query) |
| **Write speed** | Fast (just save post) | Slower (push to all followers) |
| **Storage** | No extra storage | Stores feed per user in Redis |
| **Consistency** | Always fresh | May be slightly stale |
| **Best for** | Few followers | Most users |

## 🌟 The Celebrity Problem

Fan-out on write works great for User 5 with 10 followers.  
But what about **celeb_alice** with **1.5 million followers**?

When she posts, we'd need to push that post into **1.5 million Redis sorted sets**.  
That's called **write amplification** — and it's a huge problem.

```
celeb_alice posts a photo
       │
       ▼
Push to 1,500,000 feeds!  ← This takes seconds or minutes
       │                     and hammers Redis with writes
       ▼
feed:1, feed:2, feed:3, ... feed:1500000
```

The numbers at Instagram's scale:
- Cristiano Ronaldo: 600M+ followers
- One post → 600 million Redis writes
- That's **insane**

In [ ]:
# Let's demonstrate the celebrity problem with our data
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# How many followers does each celebrity have in our demo?
cur.execute("""
    SELECT u.username, u.display_name, u.follower_count,
           COUNT(f.follower_id) AS actual_followers_in_db
    FROM users u
    LEFT JOIN follows f ON f.followee_id = u.id
    WHERE u.is_celebrity = TRUE
    GROUP BY u.id
    ORDER BY u.follower_count DESC
""")

print("Celebrity follower counts:")
print(f"{'Username':<15} {'Display Name':<25} {'Real Followers':>15} {'Demo DB':>10}")
print("-" * 70)
for row in cur.fetchall():
    print(f"{row['username']:<15} {row['display_name']:<25} {row['follower_count']:>15,} {row['actual_followers_in_db']:>10}")

print("\n⚠️  In production, fan-out on write for a celebrity post would mean")
print("   writing to MILLIONS of Redis keys — unacceptable!")

conn.close()

## 🔀 Strategy 3: Hybrid Approach (What Instagram Uses)

The solution is to **combine both strategies**:

- **Regular users** (< 100K followers): fan-out on write (push to followers' feeds)
- **Celebrities** (≥ 100K followers): fan-out on read (merge at read time)

When a user opens their feed:
1. Read precomputed feed from Redis (posts from regular users they follow)
2. Query recent posts from celebrities they follow (fan-out on read)
3. Merge both lists, sort by time

```
User opens feed
      │
      ├──► Redis: get precomputed feed    ──┐
      │    (regular users' posts)            │
      │                                      ├──► Merge + Sort ──► Return
      └──► DB: get celebrity posts         ──┘
           (fan-out on read, just a few)
```

In [ ]:
CELEBRITY_THRESHOLD = 100000  # Users with >= 100K followers are "celebrities"

def create_post_hybrid(author_id: int, caption: str):
    """
    Hybrid post creation:
    - Always save to DB
    - Only fan-out to followers if the author is NOT a celebrity
    """
    conn = get_db()
    cur = conn.cursor()

    # Check if this user is a celebrity
    cur.execute("SELECT follower_count FROM users WHERE id = %s", (author_id,))
    follower_count = cur.fetchone()[0]
    is_celebrity = follower_count >= CELEBRITY_THRESHOLD

    # Save the post
    cur.execute(
        """INSERT INTO posts (author_id, caption, media_type, media_key, created_at)
           VALUES (%s, %s, 'photo', %s, NOW())
           RETURNING id, extract(epoch from created_at)""",
        (author_id, caption, f"photos/user_{author_id}/hybrid_post.jpg")
    )
    post_id, post_ts = cur.fetchone()
    post_ts = float(post_ts)  # Decimal → float for Redis
    conn.commit()
    conn.close()

    if is_celebrity:
        print(f"👑 Celebrity post #{post_id} — skipping fan-out (too many followers)")
        print(f"   {follower_count:,} followers would be too expensive to push to")
    else:
        print(f"👤 Regular user post #{post_id} — fanning out to followers")
        fan_out_on_write(author_id, post_id, post_ts)

    return post_id, is_celebrity

# Regular user posts → fan-out happens
print("=" * 50)
print("Regular user (User 3) posts:")
print("=" * 50)
create_post_hybrid(author_id=3, caption="Regular user post!")

print()

# Celebrity posts → no fan-out
print("=" * 50)
print("Celebrity (celeb_bob, user 52) posts:")
print("=" * 50)
create_post_hybrid(author_id=52, caption="Celebrity announcement!")

In [ ]:
def get_feed_hybrid(user_id: int, limit: int = 20) -> list:
    """
    Hybrid feed retrieval:
    1. Get precomputed feed from Redis (regular users' posts)
    2. Get recent posts from celebrities we follow (fan-out on read)
    3. Merge and sort
    """
    r = get_redis()
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    start = time.time()

    # ── Part 1: Precomputed feed from Redis ──────────────────
    feed_key = f"feed:{user_id}"
    precomputed_ids = r.zrevrange(feed_key, 0, limit - 1)
    redis_time = (time.time() - start) * 1000

    # ── Part 2: Celebrity posts (fan-out on read) ────────────
    cur.execute("""
        SELECT p.id, p.author_id, p.caption, p.like_count, p.created_at
        FROM posts p
        JOIN follows f ON f.followee_id = p.author_id
        JOIN users u ON u.id = p.author_id
        WHERE f.follower_id = %s
          AND u.follower_count >= %s
        ORDER BY p.created_at DESC
        LIMIT %s
    """, (user_id, CELEBRITY_THRESHOLD, limit))
    celebrity_posts = cur.fetchall()
    celeb_time = (time.time() - start) * 1000 - redis_time

    # ── Part 3: Hydrate precomputed post IDs ─────────────────
    precomputed_posts = []
    if precomputed_ids:
        cur.execute("""
            SELECT id, author_id, caption, like_count, created_at
            FROM posts WHERE id = ANY(%s)
        """, ([int(pid) for pid in precomputed_ids],))
        precomputed_posts = cur.fetchall()

    # ── Part 4: Merge and sort ───────────────────────────────
    # Deduplicate (a celebrity post might appear in both lists)
    seen_ids = set()
    merged = []
    for post in list(precomputed_posts) + list(celebrity_posts):
        if post["id"] not in seen_ids:
            seen_ids.add(post["id"])
            merged.append(post)

    merged.sort(key=lambda p: p["created_at"], reverse=True)
    feed = merged[:limit]
    total_time = (time.time() - start) * 1000

    conn.close()

    print(f"📊 Hybrid feed stats:")
    print(f"   Redis (precomputed): {redis_time:.1f}ms → {len(precomputed_ids)} post IDs")
    print(f"   DB (celebrity read): {celeb_time:.1f}ms → {len(celebrity_posts)} posts")
    print(f"   Total: {total_time:.1f}ms → {len(feed)} posts returned")

    return feed

# Get hybrid feed for User 1
print("Loading User 1's feed (hybrid approach)...\n")
hybrid_feed = get_feed_hybrid(user_id=1, limit=10)
if hybrid_feed:
    print(f"\nTop posts in hybrid feed:")
    for i, post in enumerate(hybrid_feed, 1):
        print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}")

## 📊 Populating Redis Feeds from Existing Data

Our database already has precomputed feed entries (from `init.sql`).  
Let's load them into Redis so the hybrid read path works fully.

In [ ]:
def populate_redis_feeds():
    """
    Load precomputed feeds from PostgreSQL into Redis sorted sets.
    In production, Redis would be the primary feed store.
    """
    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    # Get all precomputed feed entries for non-celebrity authors
    cur.execute("""
        SELECT pf.user_id, pf.post_id, extract(epoch from pf.post_created_at) as ts
        FROM precomputed_feed pf
        JOIN users u ON u.id = pf.post_author_id
        WHERE u.follower_count < %s
    """, (CELEBRITY_THRESHOLD,))

    rows = cur.fetchall()
    pipeline = r.pipeline()
    for user_id, post_id, ts in rows:
        pipeline.zadd(f"feed:{user_id}", {str(post_id): float(ts)})
    pipeline.execute()

    conn.close()
    print(f"✅ Loaded {len(rows)} feed entries into Redis")

    # Show some stats
    sample_users = [1, 5, 10, 25]
    for uid in sample_users:
        count = r.zcard(f"feed:{uid}")
        print(f"   feed:{uid} has {count} posts")

populate_redis_feeds()

# Now try the hybrid feed again
print("\n" + "=" * 50)
print("Hybrid feed for User 1 (with Redis populated):")
print("=" * 50 + "\n")
hybrid_feed = get_feed_hybrid(user_id=1, limit=10)
if hybrid_feed:
    print(f"\nTop posts:")
    for i, post in enumerate(hybrid_feed, 1):
        print(f"  {i}. Post #{post['id']} by user {post['author_id']}: {post['caption'][:50]}")

## 🧠 Key Takeaways

1. **Fan-out on Read** — simple but slow at scale (N+1 queries per feed load)
2. **Fan-out on Write** — fast reads (single Redis lookup) but expensive writes for popular users
3. **The Celebrity Problem** — a user with 600M followers can't fan-out on write
4. **Hybrid approach** — fan-out on write for regular users, fan-out on read for celebrities
5. **Redis sorted sets** — perfect data structure for precomputed feeds (ZADD, ZREVRANGE)

### Interview Tips

- Start with fan-out on read (show you understand the simple approach)
- Explain why it doesn't scale (do the math: 500M users × 500 follows × 5 refreshes/day)
- Propose fan-out on write as the improvement
- Identify the celebrity problem before the interviewer asks
- Land on the hybrid approach — this is what Instagram actually uses

### What's Next?

In **Notebook 3**, we'll explore **Stories** — Instagram's ephemeral content  
that disappears after 24 hours. We'll use Redis TTL to handle automatic expiration.